In [1]:
!python3 -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 11.9 MB/s  0:00:01 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
pip install rake-nltk

Note: you may need to restart the kernel to use updated packages.


In [10]:
import json
import re
from collections import Counter
from tqdm import tqdm
import pandas as pd
import spacy
from rake_nltk import Rake
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
nltk.download('punkt') 
nltk.download('punkt_tab')



[nltk_data] Downloading package stopwords to /Users/hanaa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/hanaa/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/hanaa/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
import requests

url = "https://huggingface.co/datasets/garethpaul/children-stories-dataset/resolve/main/train.jsonl"
r = requests.get(url)
with open("train.jsonl", "wb") as f:
    f.write(r.content)

# Load stories safely
stories = []
with open("train.jsonl", "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        try:
            data = json.loads(line)
            stories.append(data)
        except json.JSONDecodeError:
            print(f"Skipping broken JSON at line {line_num}")
        except Exception as e:
            print(f"Skipping line {line_num} due to {e}")

print(f"Total stories loaded: {len(stories)}")

Total stories loaded: 9970


In [5]:
nlp = spacy.load("en_core_web_sm")

In [6]:
def extract_characters(text, top_n=3):
    """Extract main characters using spaCy NER"""
    doc = nlp(text)
    people = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    freq = Counter(people)
    return [name for name, _ in freq.most_common(top_n)]

def extract_animals_and_objects(text):
    """Extract common nouns that might be animals or objects"""
    doc = nlp(text)
    nouns = [token.text.lower() for token in doc if token.pos_ in ("NOUN", "PROPN")]
    # remove stopwords
    nouns = [w for w in nouns if w not in stop_words]
    return list(set(nouns))

def extract_themes(text, top_n=5):
    """Extract theme keywords using RAKE"""
    rake = Rake(stopwords=stop_words)
    rake.extract_keywords_from_text(text)
    return rake.get_ranked_phrases()[:top_n]

def segment_story(text, max_len=800):
    """Segment story for embedding input"""
    cleaned = re.sub(r"\s+", " ", text).strip()
    return cleaned[:max_len]

def create_feature_text(story):
    """Combine extracted features into a single string for embedding"""
    text = story.get("text", "")
    characters = extract_characters(text)
    animals_objects = extract_animals_and_objects(text)
    themes = extract_themes(text)

    features = []
    if characters:
        features.append("characters: " + ", ".join(characters))
    if animals_objects:
        features.append("animals_objects: " + ", ".join(animals_objects))
    if themes:
        features.append("themes: " + ", ".join(themes))

    return " | ".join(features)

In [7]:
def create_feature_text_safe(story):
    text = story.get("text", "").strip()
    if not text:
        # fallback if story text is empty
        return "No content"
    
    # Extract characters
    doc = nlp(text)
    characters = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    
    # Extract nouns (potential animals or objects)
    nouns = [token.text.lower() for token in doc if token.pos_ in ("NOUN", "PROPN") and token.text.lower() not in stop_words]
    
    # Extract simple RAKE keywords
    rake = Rake(stopwords=stop_words)
    rake.extract_keywords_from_text(text)
    themes = rake.get_ranked_phrases()[:5]
    
    # Combine features; if everything empty, fallback to first 300 chars of text
    features = []
    if characters:
        features.append("characters: " + ", ".join(characters))
    if nouns:
        features.append("nouns: " + ", ".join(nouns))
    if themes:
        features.append("themes: " + ", ".join(themes))
    
    if not features:
        features.append(text[:300])
    
    return " | ".join(features)

In [8]:
import sys
print(sys.executable)

/Library/Frameworks/Python.framework/Versions/3.13/bin/python3


In [11]:
from rake_nltk import Rake
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

rake = Rake(stopwords=stop_words)
example_text = stories[0]['text']
rake.extract_keywords_from_text(example_text)
print(rake.get_ranked_phrases()[:5])

['magic bell makes snack time extra fun ," explained teacher rabbit', 'sweet maya ," teacher rabbit said softly', 'teacher rabbit noticed maya sitting alone', 'penny pig made silly slurping sounds', 'friend sophie squirrel made chomping sounds']


In [12]:
feature_texts = [create_feature_text_safe(s) for s in stories]
titles = [s['title'] for s in stories]
print(f"Feature texts generated: {len(feature_texts)}")

Feature texts generated: 9970


In [13]:
model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(feature_texts, batch_size=8, show_progress_bar=True)
print(f"Embeddings shape: {len(embeddings)} x {len(embeddings[0])}")

Batches: 100%|██████████████████████████████| 1247/1247 [12:15<00:00,  1.70it/s]

Embeddings shape: 9970 x 768


In [14]:
NUM_CLUSTERS = 8
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
cluster_ids = kmeans.fit_predict(embeddings)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
cluster_labels = []

for cluster_num in range(NUM_CLUSTERS):
    idx = [i for i, c in enumerate(cluster_ids) if c == cluster_num]
    combined_text = " ".join([feature_texts[i] for i in idx]).lower()
    words = re.findall(r"[a-zA-Z]+", combined_text)
    filtered_words = [w for w in words if w not in stop_words]
    freq = pd.Series(filtered_words).value_counts().head(8)
    keywords = list(freq.index)
    cluster_labels.append(", ".join(keywords))

In [16]:
df = pd.DataFrame({
    "title": titles,
    "feature_text": feature_texts,
    "cluster_id": cluster_ids,
    "cluster_keywords": [cluster_labels[c] for c in cluster_ids]
})

# Count stories per cluster
cluster_counts = df['cluster_id'].value_counts().sort_index()
print("\nStories per cluster:")
for i, count in cluster_counts.items():
    print(f"Cluster {i}: {count} stories — Keywords: {cluster_labels[i]}")



Stories per cluster:
Cluster 0: 623 stories — Keywords: joey, roo, kip, mama, koko, kori, kiki, pip
Cluster 1: 2105 stories — Keywords: leo, emma, rocco, bella, mama, eyes, max, time
Cluster 2: 2124 stories — Keywords: mama, shelly, leo, emma, melody, eyes, grandma, ellie
Cluster 3: 515 stories — Keywords: ollie, finn, mama, oliver, paws, otter, pip, eyes
Cluster 4: 269 stories — Keywords: nutkin, nutmeg, squirrel, pip, friends, paws, tail, eyes
Cluster 5: 908 stories — Keywords: benny, mama, bear, paws, friends, time, eyes, rosie
Cluster 6: 2388 stories — Keywords: mia, pip, tap, tiko, miko, mama, whiskers, eyes
Cluster 7: 1038 stories — Keywords: pip, penny, pippa, poppy, mama, friends, eyes, tap


In [17]:
dropdown_options = [
    {"cluster_id": i, "label": cluster_labels[i]} for i in range(NUM_CLUSTERS)
]

with open("dropdown_clusters.json", "w", encoding="utf-8") as f:
    json.dump(dropdown_options, f, indent=2)

print("\nDropdown JSON saved as 'dropdown_clusters.json'")


Dropdown JSON saved as 'dropdown_clusters.json'


In [18]:
import json

# Load the JSON file
with open("dropdown_clusters.json", "r", encoding="utf-8") as f:
    dropdown_data = json.load(f)

# Inspect the first few items
print(dropdown_data[:5])

[{'cluster_id': 0, 'label': 'joey, roo, kip, mama, koko, kori, kiki, pip'}, {'cluster_id': 1, 'label': 'leo, emma, rocco, bella, mama, eyes, max, time'}, {'cluster_id': 2, 'label': 'mama, shelly, leo, emma, melody, eyes, grandma, ellie'}, {'cluster_id': 3, 'label': 'ollie, finn, mama, oliver, paws, otter, pip, eyes'}, {'cluster_id': 4, 'label': 'nutkin, nutmeg, squirrel, pip, friends, paws, tail, eyes'}]
